### Widok listy filmów

Udajmy się do [`movies/views.py`](http://localhost:8888/edit/movies/views.py) aby utworzyć widok listy filmów.

Na dobry początek trzeba zaimportować model (mądre IDE jak PyCharm lub VSCode same to zaproponują/zrobią).
Następnie piszemy kolejną funkcję, która przyjmuje zapytanie (`request`) jako argument.

```python
from movies.models import Movie  # NOWE

def list_movies(request):
    movies = Movie.objects.all()
    return render(
        request, 
        template_name="movie_list.html", 
        context={"movies": movies}
    )  # NOWE
```

W tym miejscu wiele się dzieje! Zacznijmy od góry:

* dzięki zaimportowaniu modelu `Movie` będziemy mogli wejść w interakcje z bazą danych za pośrednictwem Django, czyli między innymi:
  * dodawać nowe obiekty typu `Movie`
  * edytować i usuwać istniejące obiekty `Movie`
  * odczytywać a także filtrować istniejące obiekty `Movie`

* `objects` - każdy model w Django ma coś co się nazywa [manager](https://docs.djangoproject.com/en/5.2/topics/db/managers/), nie będziemy wchodzić tutaj w szczegóły, ale jest to interfejs przez który dostarczane są nam operacje na bazie danych. W praktyce, umożliwia to nam tworzenie, edycję, usuwanie i inne zapytania do bazy danych.

* `Movie.objects.all()` użyje menedżera obiektów typu `Movie` i zapyta bazę danych o WSZYSTKIE filmy. Dostaniemy więc listę całej zawartości naszej filmoteki.

* Do wygenerowania odpowiedzi aplikacji zostanie użyty szablon `movie_list.html` (tak, musimy go teraz stworzyć!)

* Lista filmów zostanie dodana do kontekstu HTML

### Rejestracja podstrony listy filmów

Aby widok mógł być wyświetlony trzeba mu przydzielić jakiś adres. Udajmy się więc do [`urls.py`](http://localhost:8888/edit/goodmovies/urls.py) czyli o URL resolvera naszego projektu. Tutaj należy dodać `path()`, który będzie kierował zapytania przeglądarek użytkowników z konkretnego adresu na stronę powstającej listy filmów.

Przykładowo, do listy `urlpatterns` dodajmy linijkę:

```python
urlpatterns = [
    ...
    
    # http://127.0.0.1:8000/filmy/
    path('filmy/', views.list_movies),  # NOWE
]
```

Teraz link [http://127.0.0.1:8000/filmy/](http://127.0.0.1:8000/filmy/) powinien nas kierować do listy filmów, którą przed chwilą napisaliśmy w [`views.py`](http://localhost:8888/edit/movies/views.py). Został jeszcze jeden element - szablon HTML, który wyświetli dane.

### Szablon HTML listy

Stwórzmy plik o nazwie [`movie_list.html`](http://localhost:8888/edit/movies/templates/movie_list.html) (oczywiście musi być umieszczony w katalogu szablonów naszej aplikacji [`movies/templates/`](movies/templates/)).

In [8]:
!touch movies/templates/movie_list.html

Na początek możemy sprawdzić, czy wszystko dobrze zaprogramowaliśmy. Spróbujmy więc wypisać zmienną `movies`, którą dodaliśmy do kontekstu szablonu.

```django
{{ movies }}
```

Jeśli w przeglądarce pojawiła się lista obiektów z tajemniczym `QuerySet`, to jesteśmy na dobrej drodze.

Aby wypisać elementy list prostą pętlą można posłużyć się tagiem Django

`{% for %}`.

```django
{% for movie in movies %}
<p>
    Film: "{{ movie }}"
</p>
{% endfor %}
```

Powyższy kawałek kodu prze-iteruje po liście, którą dostarczyliśmy z widoku. Następnie dla każdego elementu wstawi HTML zawarty w środku, czyli w naszym przypadku akapit z napisem `Film: "{{ movie }}"`.

## Dalsze prace

1. Wypisz osobno każde z pól modelu filmu np.: tytuł (`movie.title`), datę publikacji (`movie.published_at`)
1. Wykorzystaj HTML (np.: `<b></b>`, `<i></i>`) i CSS (np.: `<style>p {color: green;}</style>`) do poprawienia wyglądu listy
1. Przebuduj szablon `movie_list.html` tak aby wykorzystywał (rozszerzał) szablon bazowy `base.html` (np.: `{% extends "base.html" %}{% block content %}{% endblock %}`)
1. Dodaj widok szczegółów pierwszego, pojedynczego filmu (np.: `Movie.objects.all()[0]`)

In [ ]:
pip uninstall -y Pillow

# Django - Rozbudowa modelu danych 1
*[Mikołaj Leszczuk](mailto:mikolaj.leszczuk@agh.edu.pl), [Agnieszka Rudnicka](mailto:rudnicka@agh.edu.pl)*

* Relacyjne bazy danych
  * Problem z polem reżysera w obecnym modelu
  * Relacje i klucz obcy
  * Graficzna reprezentacja modelu danych

* Ulepszamy nasze modele
  * Model reżysera
    * Pole z obrazem ImageField
    * Etykieta verbose name
    * Meta-dane modelu
  * Poprawki w modelu filmów
    * Tworzymy migracje
    * Aplikujemy migrację
    * Konfiguracja plików MEDIA
    * Kontrola
    * Rejestrujemy model reżysera w panelu administracyjnym
    * Tworzymy reżyserów przez panel administratora

## Relacyjne bazy danych

Do tej pory nasza aplikacja posiadała dość prosty i ubogi model danych. Była to tylko pojedyncza tabela przechowująca filmy (`class Movie(models.Model)`: w pliku [`movies/models.py`](http://localhost:8888/edit/movies/models.py)).

Pora na utworzenie oddzielnego modelu na reżyserów i recenzje oraz rozszerzenie istniejącego modelu filmów o dodatkowe pola.

### Problem z polem reżysera w obecnym modelu

Wróćmy do pliku z naszymi modelami, [`movies/models.py`](http://localhost:8888/edit/movies/models.py). Gdybyśmy umieścili pole na na reżysera:

```python
class Movie(models.Model):
    ...
    director = models.CharField(null=True, max_length=128)
```

to napotkamy następujące problemy:

* jeśli mamy kilka filmów tego samego reżysera, to informacje o nim się powtarzają w wielu wpisach (redundancja
danych), *mówiąc prościej - marnotrawstwo miejsca przez powtarzanie tych samych informacji*

* jest większa szansa popełnienia błędu i powstania różnych zapisów tego samego imienia/nazwiska, co później może się przekładać na problemy z wyszukaniem wszystkich filmów danego reżysera

* przy aktualizacji informacji o reżyserze trzeba zaktualizować wszystkie filmy, które reżyserował, co się przekłada na wiele edycji zamiast jednej

* i nie tylko :)

### Relacje i klucz obcy

Jedną z podstawowych zalet baz danych których używamy (SQLite, PostgreSQL i innych) jest możliwość tworzenia relacji między modelami.

Biorąc na warsztat przykład z autorem i książkami, rozwiązaniem jakie często znajdziemy w praktyce jest wydzielenie
osobnej tabeli na dane o autorach. W ten sposób będziemy mieli książki w jednej tabeli a autorów w drugiej. Unikniemy redundancji danych, problemów z aktualizowaniem wielu wpisów i innych.

Bazy relacyjne pozwalają zdefiniowanie specjalnego pola, w którym będzie przechowywany identyfikator wiersza z innej, "obcej" tabeli. Stąd też polska nazwa "klucz obcy" i angielska "foreign key".

Ten klucz obcy, to nic innego jak pole `id`, które jest automatycznie definiowane przez framework Django dla każdego modelu (jeśli użytkownik nie zdefiniuje własnego). `id` jest też często nazywany "kluczem głównym". Oznacza to, że identyfikuje on rekordy/wiersze będąc unikatowym i jednoznacznym. Tak jak numer PESEL się nie powtarza i zawsze identyfikuje jednego człowieka, tak **klucz główny identyfikuje jeden i tylko jeden wpis w tabeli bazy danych**.

### Graficzna reprezentacja modelu danych

To jak przechowujemy dane w bazie często jest o wiele bardziej skomplikowane niż jedna, czy dwie tabele. Aby było łatwiej zrozumieć co z czym się łączy tworzy się modele danych np przy pomocy UML (Unified Modeling Language).

Poniższy diagram UML przedstawia aplikacje przygotowaną z myślą o zarządzaniem biblioteką [[źródło](https://developer.mozilla.org/en-US/docs/Learn_web_development/Extensions/Server-side/Django/Models)].

![](https://developer.mozilla.org/en-US/docs/Learn_web_development/Extensions/Server-side/Django/Models/local_library_model_uml.svg)

## Ulepszamy nasze modele

Stwórzmy osobny model na reżysera i wykorzystajmy mechanizmy baz relacyjnych do stworzenia powiązań z filmami.

### Model reżysera

W pliku [`movies/models.py`](http://localhost:8888/edit/movies/models.py) gdzie opisaliśmy uprzednio model filmu, dodajmy teraz model reżysera:

```python
class Director(models.Model):
    first_name = models.CharField(verbose_name="imię", max_length=100)
    last_name = models.CharField(verbose_name="nazwisko", max_length=100)
    about = models.TextField(verbose_name="o reżyserze", blank=True)
    photo = models.ImageField(verbose_name="zdjęcie", blank=True)
    
    class Meta:
        ordering = ["last_name", "first_name"]
        verbose_name = "reżyser"
        verbose_name_plural = "reżyserzy"
        
    def __str__(self):
        return "Reżyser: " + self.first_name + " " + self.last_name
```

#### Pole z obrazem ImageField

Mamy tutaj model zawierający pole na imię, nazwisko, opis oraz zdjęcie (obraz graficzny). To ostatnie jest szczególnie ciekawe. Django pozwala nam tworzyć pola, które przechowują ścieżkę do pliku. Rzadko kiedy przechowuje się pliki wgrane przez użytkowników w bazie danych. Django domyślnie w polu `ImageField` przechowuje informacje o nazwie pliku. Ustawienia gdzie tych plików szukać będą już specyficzne dla projektu, czasem nawet serwera.

#### Etykieta `verbose_name`

Kolejna nowością jest `verbose_name`, które zostało zdefiniowane dla każdego pola. Jest to coś w rodzaju domyślnej etykiety wyświetlanej jeśli żadna inna nie została zapewniona. Dzięki temu zabiegowi, w panelu administratora zamiast angielskich nazw zmiennych zobaczymy polskie etykiety.

#### Meta-dane modelu

Ostatnim dodatkiem jest podklasa `class Meta`. Tutaj definiuje się meta-dane dotyczące modelu, między innymi domyślne sortowanie elementów. Pozwala to zapewnić kolejność zwracanych danych z bazy, co jest szczególnie ważne przy widokach z paginacją/stronicowaniem. No i wprowadza odrobinę ładu, bo sortowanie po nazwisku/imieniu jest dla nas (ludzi) naturalne.

### Poprawki w modelu filmów

Skoro już wzbogacamy nasz model o etykiety, dodajmy je również do modelu filmu:

```python
class Movie(models.Model):
    title = models.CharField(verbose_name="tytuł", max_length=100)
    short_description = models.TextField(verbose_name="opis")
    published_at = models.DateField(verbose_name="data premiery")

    # !!!
    director = models.ForeignKey(
        to="movies.Director",
        verbose_name="reżyser",
        related_name="movies",
        on_delete=models.CASCADE,
        null=True
    )

    class Meta:
        ordering = ["title"]
        verbose_name = "film"
        verbose_name_plural = "filmy"

    def __str__(self):
        return "Film: " + self.title
```

Przy okazji musimy zmienić również pole `director`. Będzie ono teraz kluczem obcym, co jest odzwierciedlone za pomocą
pola `ForeignKey()` w frameworku Django.

#### Tworzymy migracje

Na początek wykonajmy polecenie, które pozwala podglądnąć jaka migracja będzie wygenerowana. Niestety zakończy się
ono błędem:

In [ ]:
!python3 manage.py makemigrations --dry-run -v 3

Doinstalujmy więc bibliotekę wymaganą do obsługi obrazów i pół `ImageField`.

In [ ]:
!pip3 install Pillow

**Nie zapomnijmy dodać też biblioteki do pliku wymagań [`requirements.txt`](http://localhost:8888/edit/requirements.txt)!**

Teraz polecenie `python3 manage.py makemigrations` powinno się pomyślnie wykonać.

In [ ]:
!python3 manage.py makemigrations

#### Aplikujemy migrację

Żeby odpowiednie tabele zostały utworzone, a istniejące zaktualizowane trzeba standardowo zaaplikować migrację.

In [ ]:
!python3 manage.py migrate

#### Konfiguracja plików MEDIA

Zanim sprawdzimy nasze zmiany dodajmy jeszcze konfigurację gdzie mają być wgrywane pliki "media", czyli obrazy
wgrane do pól typu `FileField` oraz `ImageField`.

Na samym końcu pliku [`goodmovies/settings.py`](http://localhost:8888/edit/goodmovies/settings.py) w katalogu konfiguracji projektu odszukajmy linijki dotyczące plików statycznych i dodajmy dwie z `MEDIA_URL` i `MEDIA_ROOT`:

```python
# Static files (CSS, JavaScript, Images)
# https://docs.djangoproject.com/en/5.2/howto/static-files/

STATIC_URL = 'static/'

MEDIA_URL = '/media/'  # <-- nowe
MEDIA_ROOT = BASE_DIR / 'media'  # <-- nowe
```

Dzięki temu pliki statyczne będą wgrywane do podkatalogu `/media` w głównym folderze z naszymi plikami projektu.

Aby Django mogło udostępniać pliki z katalogu `media/` podczas pracy w trybie developerskim (czyli gdy `DEBUG = True`), musimy dodać odpowiednią konfigurację w pliku [`goodmovies/urls.py`](http://localhost:8888/edit/goodmovies/urls.py). Na samym dole pliku dopisujemy:

```python
from django.conf import settings
from django.conf.urls.static import static

if settings.DEBUG:
    urlpatterns += static(settings.MEDIA_URL, document_root=settings.MEDIA_ROOT)
```

Dzięki temu Django automatycznie obsłuży żądania do adresów rozpoczynających się od `/media/` i będzie zwracać odpowiadające im pliki z katalogu `media/` w naszym projekcie.

To rozwiązanie działa tylko w trybie deweloperskim - w środowisku produkcyjnym za obsługę plików statycznych i multimedialnych powinien odpowiadać np. nginx.

#### Kontrola

[Uruchommy aplikację](/terminals/1) (`python3 manage.py runserver`) i zaglądnijmy do panelu administracyjnego ([http://127.0.0.1:8000/admin](http://127.0.0.1:8000/admin)).

Na pierwszy rzut oka niewiele się zmieniło. Są jedynie etykiety po polsku. Niestety nie ma śladu po modelu reżysera. Ale czy na pewno? Zobaczmy widok tworzenia nowego filmu.

Pojawiło się pole wielokrotnego wyboru, jednak nie można nic wybrać ani dodać... bowiem nie zarejestrowaliśmy naszego nowego modelu jako edytowalnego przez panel administracyjny.

#### Rejestrujemy model reżysera w panelu administracyjnym

Udajmy się do [`movies/admin.py`](http://localhost:8888/edit/movies/admin.py).

```python
from django.contrib import admin

# Register your models here.
from movies.models import Movie, Director # Director dodany

admin.site.register(Movie)
admin.site.register(Director) # nowe
```

Zapiszmy plik i odświeżmy stronę w przeglądarce (z panelem administracyjnym). Zaraz poniżej filmów powinna się pojawić sekcja z reżyserami.